# 05: HITL 3D 目視確認 / Human-in-the-Loop 3D Review

04_compound_selection で選択した化合物のドッキングポーズを 3D で目視確認し、
Pass / Wait / Fail の判定を記録します。

Review docked poses in 3D and record Pass/Wait/Fail decisions for selected compounds.

In [ ]:
from pathlib import Path
from datetime import datetime

import pandas as pd
from rdkit import Chem

from docking_analysis.visualization.py3dmol_viewer import (
    show_complex,
    show_pose_comparison,
    export_review_results,
)

from docking_analysis.visualization.pymol import write_reference_overlay_pml
from docking_analysis.visualization.contact_heatmap import plot_contact_heatmap
from docking_analysis.analysis.validation import compute_artifact_score, validate_pose_in_box

## 設定 / Configuration

入力ファイルと出力先を指定してください。

Specify input files and output directory.

In [ ]:
# === ユーザー設定 / User Configuration ===
RECEPTOR_PDB = "../data/receptor_clean.pdb"  # 受容体 PDB / receptor PDB
SELECTED_SDF = "../results/selected/selected_compounds.sdf"  # 04 の出力 / output from 04
SELECTED_CSV = "../results/selected/selected_compounds.csv"
OUTPUT_DIR = Path("../results/review")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REVIEWER = "reviewer_name"  # レビュアー名 / your name

## データ読み込み / Load Data

In [ ]:
supplier = Chem.SDMolSupplier(str(SELECTED_SDF), removeHs=False)
mols = [m for m in supplier if m is not None]
df = pd.read_csv(SELECTED_CSV) if Path(SELECTED_CSV).exists() else pd.DataFrame()
print(f"Loaded {len(mols)} compounds for review")

## ポーズ品質指標サマリー / Pose Quality Metrics Summary

ひずみエネルギー・インタラクションスコア・アーティファクトスコアを表示して  
目視確認の優先順位を判断します。

Shows strain energy, interaction score and artifact score to prioritise review order.

In [ ]:
# アーティファクトスコアを計算（列が存在する場合のみ）
# Compute artifact score if relevant columns exist
artifact_score_kwargs = {}
if "centroid_distance"  in df.columns: artifact_score_kwargs["centroid_col"]    = "centroid_distance"
if "strain_h_relaxed"   in df.columns: artifact_score_kwargs["strain_col"]      = "strain_h_relaxed"
if "interaction_score"  in df.columns: artifact_score_kwargs["interaction_col"] = "interaction_score"

if artifact_score_kwargs and not df.empty:
    df["artifact_score"] = compute_artifact_score(df, **artifact_score_kwargs)
    display_cols = ["mol_name", "docking_score"] + list(artifact_score_kwargs.values()) + ["artifact_score"]
    display_cols = [c for c in display_cols if c in df.columns]
    print("Top 10 poses by artifact score (lower = better):")
    print(df[display_cols].sort_values("artifact_score").head(10).to_string(index=False))
else:
    print("Quality metric columns not found in CSV — run 03_selection.ipynb first.")

In [ ]:
# グリッドボックス内にリガンド重心が収まるか確認（config が利用可能な場合）
# Validate ligand centroid vs grid box (requires AnalysisConfig)
try:
    from docking_analysis import AnalysisConfig
    cfg = AnalysisConfig.from_toml("../notebooks/templates/project_config.toml")
    if cfg.gridbox is not None:
        print(f"Grid box centre: {cfg.gridbox.center}, size: {cfg.gridbox.size}")
        for i, mol in enumerate(mols[:5]):  # 先頭 5 件だけ確認
            result = validate_pose_in_box(mol, cfg.gridbox.center, cfg.gridbox.size)
            name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"mol_{i}"
            status = "✓ in_box" if result["in_box"] else ("△ extended" if result["in_extended"] else "✗ outside")
            print(f"  {name}: {status}  dist={result['dist_from_center']:.1f} Å")
    else:
        print("No gridbox defined in config.")
except Exception as e:
    print(f"Box check skipped: {e}")

## 選択化合物の接触ヒートマップ / Contact Heatmap for Selected Compounds

選択した化合物群の残基接触パターンをヒートマップで確認します。

Visualise residue contact patterns across selected compounds.

In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    residue_cols = [
        c for c in df.columns
        if any(c.endswith(t) for t in ["HBAcceptor","HBDonor","Hydrophobic","Anionic","Cationic","PiStacking","PiCation"])
    ]
    if residue_cols:
        # クラスター or receptor でグループ化（存在する列を優先）
        group_col = "cluster_id" if "cluster_id" in df.columns else                     "receptor"   if "receptor"   in df.columns else None
        if group_col:
            fig = plot_contact_heatmap(
                df, residue_cols, group_col=group_col,
                annotate=True,
                output_path=Path(OUTPUT_DIR) / "review_contact_heatmap.png",
            )
            plt.show()
            print(f"Saved: {Path(OUTPUT_DIR) / 'review_contact_heatmap.png'}")
        else:
            print("No cluster_id or receptor column found for grouping.")
    else:
        print("No ProLIF interaction columns found — run 01_processing.ipynb first.")
else:
    print("DataFrame is empty — load CSV in the configuration cell above.")

## PyMOL オーバーレイスクリプト生成 / Generate PyMOL Overlay Script

目視確認用にリファレンスリガンドとのオーバーレイ `.pml` スクリプトを生成します。

Generates a PyMOL overlay `.pml` script for the reviewed compounds.

In [ ]:
REFERENCE_SDF = None  # ← リファレンスリガンド SDF を指定（任意）/ reference ligand SDF

pymol_dir = OUTPUT_DIR / "pymol"
pymol_dir.mkdir(parents=True, exist_ok=True)

if REFERENCE_SDF is not None and Path(REFERENCE_SDF).exists():
    tier_col = "decision" if "decision" in df.columns else None
    overlay_pml = write_reference_overlay_pml(
        selected_sdf=SELECTED_SDF,
        reference_sdf=REFERENCE_SDF,
        protein_pdb=RECEPTOR_PDB,
        output_pml=pymol_dir / "review_overlay.pml",
        tier_col=tier_col,
        tier_colors={"Pass": "green", "Wait": "yellow", "Fail": "red"},
    )
    print(f"Overlay PML saved: {overlay_pml}")
    print("Open with: pymol review_overlay.pml")
else:
    print("Set REFERENCE_SDF to enable PyMOL overlay script generation.")

## 3D ポーズ確認 / 3D Pose Review

各化合物のタンパク質-リガンド複合体を 3D で表示します。
下のセルの `idx` を変更して化合物を切り替えてください。

View each protein-ligand complex in 3D. Change `idx` to switch compounds.

In [ ]:
idx = 0  # ← 確認する化合物のインデックス / compound index to review

if idx < len(mols):
    mol = mols[idx]
    name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"compound_{idx}"
    print(f"Reviewing: {name}")
    
    view = show_complex(RECEPTOR_PDB, mol)
    view.show()

## ポーズ比較 / Pose Comparison

複数ポーズをオーバーレイ表示して比較します。

Overlay multiple poses for comparison.

In [ ]:
compare_indices = [0, 1, 2]  # ← 比較するインデックス / indices to compare
compare_mols = [mols[i] for i in compare_indices if i < len(mols)]
compare_labels = [f"pose_{i}" for i in compare_indices]

if compare_mols:
    view = show_pose_comparison(RECEPTOR_PDB, compare_mols, labels=compare_labels)
    view.show()

## 評価記録 / Record Decisions

各化合物に対して Pass / Wait / Fail を記録してください。

Record your Pass/Wait/Fail decision for each compound.

In [ ]:
# 手動評価テンプレート / Manual review template
# 下のリストを編集して判定を記録してください
# Edit the list below to record your decisions

review_results = []

# Example:
# review_results.append({"compound_id": "mol_0", "decision": "Pass", "comment": "Good H-bond to GLN30", "reviewer": REVIEWER, "timestamp": datetime.now().isoformat()})
# review_results.append({"compound_id": "mol_1", "decision": "Fail", "comment": "Steric clash with TYR68", "reviewer": REVIEWER, "timestamp": datetime.now().isoformat()})
# review_results.append({"compound_id": "mol_2", "decision": "Wait", "comment": "Need MD simulation to confirm", "reviewer": REVIEWER, "timestamp": datetime.now().isoformat()})

print(f"Recorded {len(review_results)} decisions")

## エクスポート / Export Results

レビュー結果を CSV に出力します。

Export review decisions to CSV.

In [ ]:
if review_results:
    out_path = export_review_results(review_results, OUTPUT_DIR / "review_results.csv")
    print(f"Exported {len(review_results)} decisions to {out_path}")
    pd.DataFrame(review_results)
else:
    print("No decisions recorded yet. Edit the cell above to add reviews.")